# LAnoBERT Kaggle - BGL E0 baseline reproducible

Notebook clone trực tiếp `https://github.com/rubyhcm/lv-new.git`, tìm thư mục `LAnoBERT`, rồi chạy từng bước trên Kaggle GPU.

Notebook này chạy E0 theo paper: train chỉ trên normal BGL logs, MLM 20%, không NSP, 10 epochs, batch 32, và inference trên toàn bộ test set. Smoke config không được sử dụng trong notebook baseline này.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys, os

GITHUB_URL = 'https://github.com/rubyhcm/lv-new.git'
GITHUB_BRANCH = 'main'
CLONE_DIR = Path('/kaggle/working/lv-new')
PROJECT = Path('/kaggle/working/LAnoBERT')
CONFIG = 'configs/bgl_baseline.yaml'
DOWNLOAD_BGL = False
print('GitHub:', GITHUB_URL)
print('Baseline config:', CONFIG)
print('Working clone:', CLONE_DIR)


In [ ]:
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Hãy bật Accelerator = GPU trong Kaggle.')


In [ ]:
os.chdir('/kaggle/working')
if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_URL, str(CLONE_DIR)], check=True)
candidates = [p for p in CLONE_DIR.rglob('lanobert') if p.is_dir() and (p.parent / 'configs').is_dir()]
assert len(candidates) == 1, f'Không tìm thấy duy nhất LAnoBERT: {candidates}'
PROJECT = candidates[0].parent
print('Project:', PROJECT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip())


In [ ]:
%pip install -q -r {PROJECT / 'requirements.txt'} pytest

In [ ]:
os.chdir(PROJECT)
print('Current directory:', Path.cwd())
print('Config:', Path(CONFIG).exists())
print('BGL directory:', Path('data/BGL').exists())
if Path('data/BGL').exists():
    print(*sorted(str(p) for p in Path('data/BGL').iterdir()), sep='\n')


In [ ]:
PROJECT = Path(PROJECT).resolve()
os.chdir(PROJECT)
raw_log = PROJECT / 'data/BGL/BGL.log'
if not raw_log.exists():
    candidates = [p for p in Path('/kaggle/input').rglob('BGL.log') if p.is_file()]
    if candidates:
        source_bgl = candidates[0].parent
        target_bgl = PROJECT / 'data/BGL'
        target_bgl.mkdir(parents=True, exist_ok=True)
        for source in source_bgl.iterdir():
            target = target_bgl / source.name
            if source.is_file() and not target.exists():
                shutil.copy2(source, target)
        print('Copied BGL from:', source_bgl)
    elif DOWNLOAD_BGL:
        subprocess.run(['bash', 'scripts/download_data.sh', 'bgl'], cwd=PROJECT, check=True)
bgl_candidates = list(Path('/kaggle/input').rglob('BGL.log'))
assert raw_log.exists(), f"Thiếu {raw_log}; các file BGL tìm thấy: {bgl_candidates}"
print(raw_log, f'{raw_log.stat().st_size / (1024**3):.2f} GB')


## Kiểm tra mã nguồn

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-v'], check=True)
subprocess.run(['bash', '-n', 'scripts/ensure_data.sh', 'scripts/run_pipeline.sh'], check=True)
print('Tests and shell checks passed')


## Split và preprocess BGL

In [ ]:
subprocess.run(['bash', 'scripts/ensure_data.sh', CONFIG], check=True)

In [ ]:
for name in ['BGL_train_normal.raw', 'BGL_test.raw', 'BGL_test_label.log', 'BGL_train_normal_parsed.log', 'BGL_test_parsed.log']:
    path = Path('data/BGL') / name
    print(f'{path}: {path.stat().st_size / (1024**2):.1f} MB' if path.exists() else f'MISSING: {path}')


## Train tokenizer

In [ ]:
# Train a fresh tokenizer in the baseline-specific artifact directory.
shutil.rmtree(Path('outputs/BGL/baseline_tokenizer'), ignore_errors=True)
subprocess.run([sys.executable, '-m', 'lanobert.tokenizer', '--config', CONFIG], check=True)
print('Baseline tokenizer ready:', Path('outputs/BGL/baseline_tokenizer').exists())


## Tạo cấu hình Kaggle

In [ ]:
from lanobert.utils import load_config
from lanobert.tokenizer import load_tokenizer, vocab_path_for

baseline_cfg = load_config(CONFIG)
print(Path(CONFIG).read_text())
assert baseline_cfg.get('run_name') == 'bgl_baseline_e0'
assert int(baseline_cfg.get('train.num_train_epochs')) == 10
assert int(baseline_cfg.get('train.per_device_train_batch_size')) == 32
assert baseline_cfg.get('inference.max_eval_samples') is None
print('Baseline configuration validated: 10 epochs, batch 32, full evaluation.')


## Train LAnoBERT baseline

In [ ]:
from collections import Counter
from lanobert.tokenizer import load_tokenizer, vocab_path_for
from lanobert.utils import load_config

baseline_cfg = load_config(CONFIG)
vocab_file = vocab_path_for(baseline_cfg)
train_file = Path(baseline_cfg.get_path('paths.train_normal'))
test_file = Path(baseline_cfg.get_path('paths.test_log'))
label_file = Path(baseline_cfg.get_path('paths.test_label'))
tokenizer = load_tokenizer(vocab_file, max_len=512)

print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Baseline config:', Path(CONFIG).resolve())
print('Tokenizer:', Path(vocab_file).resolve())
print('Tokenizer vocab size:', tokenizer.vocab_size)
print('Train corpus:', train_file)
print('Train corpus size MB:', train_file.stat().st_size / (1024**2))
assert tokenizer.vocab_size == 1000
assert train_file.is_file() and train_file.stat().st_size > 0

with train_file.open(encoding='utf-8') as handle:
    sample_lines = [line.strip() for _, line in zip(range(8), handle) if line.strip()]
assert sample_lines
encoded = tokenizer(sample_lines, add_special_tokens=False)['input_ids']
content_ids = [token_id for ids in encoded for token_id in ids if token_id not in {
    tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.cls_token_id,
    tokenizer.sep_token_id, tokenizer.mask_token_id,
}]
print('Sample line:', sample_lines[0])
print('Sample content token count:', len(content_ids))
assert content_ids

# Validate MLM labels without materializing the full training dataset in the notebook.
from transformers import DataCollatorForLanguageModeling
encoded_probe = tokenizer(sample_lines, truncation=True, max_length=512, return_special_tokens_mask=True)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.20)
probe = collator([{
    'input_ids': ids,
    'attention_mask': mask,
    'special_tokens_mask': special,
} for ids, mask, special in zip(
    encoded_probe['input_ids'], encoded_probe['attention_mask'], encoded_probe['special_tokens_mask'])])
print('Masked labels in probe batch:', int((probe['labels'] != -100).sum()))
assert (probe['labels'] != -100).any()

labels = label_file.read_text(encoding='utf-8').split()
label_counts = Counter(labels)
print('Test labels:', label_counts)
assert label_counts == Counter({'0': 903310, '1': 348460})
assert test_file.is_file() and sum(1 for _ in test_file.open(encoding='utf-8')) == 1251770
print('Baseline pre-training validation OK: full corpus and full-label test set.')


In [ ]:
# Official E0 baseline: full normal training corpus, 10 epochs, batch 32.
shutil.rmtree(Path('outputs/BGL/baseline_model'), ignore_errors=True)
subprocess.run([sys.executable, '-m', 'lanobert.train', '--config', CONFIG], check=True)


## Inference và xem kết quả

In [ ]:
# Score the complete BGL test set (903,310 normal + 348,460 anomalous).
subprocess.run([sys.executable, '-m', 'lanobert.inference', '--config', CONFIG], check=True)
result_dir = Path('outputs/BGL/baseline_results')
print(*sorted(str(p) for p in result_dir.iterdir()), sep='\n')


In [ ]:
for report in sorted(Path('outputs/BGL/baseline_results').glob('*report.txt')):
    print(f'\n===== {report.name} =====\n{report.read_text()[:3000]}')
archive = shutil.make_archive('/kaggle/working/lanobert-bgl-baseline-e0', 'zip', 'outputs/BGL')
print('Baseline artifact:', archive)
